In [1]:
# Original code for KdV: 
# https://github.com/ketch/AMCS-394D-2023/blob/main/in-class%20notebooks/Dispersive%20traveling%20waves.ipynb
import numpy as np
import matplotlib.pyplot as plt
import scipy.integrate
from ipywidgets import interact, FloatSlider
import matplotlib
font = {'size'   : 15}
matplotlib.rc('font', **font)

In this notebook, we look for solitary ground state solutions of the NLSH:
$$
\begin{align}
v_x = -\mu \tau p+p,\\
p_x = -\kappa v^3+\mu v, 
\end{align}
$$
where $\mu$ is a propagation constant.


In [2]:
fig, axes = plt.subplots(1,2,figsize=(15,5),dpi=300)
plt.close()

def NLSH(mu=1.0,tau=0.01,kappa=10.,v0=1.5,xmax=20.):

    v0=0.#np.sqrt(mu/kappa)

    axes[0].cla()
    axes[1].cla()

    v = np.linspace(-1.5, 1.5, 50)
    p = np.linspace(-1, 1., 50)

    V, P = np.meshgrid(v, p)

    dv = P-mu*tau*P
    dp = -kappa*V**3+mu*V

    stream = axes[0].streamplot(V,P,dv,dp,broken_streamlines=False,density=0.8)
    axes[0].set_xlabel('v'); axes[0].set_ylabel('p');
    axes[0].axis('image')
    
    def rhs(t,w):
        v,p = w
        return np.array([p-mu*tau*p,-kappa*v**3+mu*v])

    w0 = np.array([v0,-0.001])
    w01 = np.array([-v0,0.001])
    t_eval = np.linspace(0,xmax,1000)
    t_eval1 = np.linspace(xmax,2*xmax,1000)
    forwardsoln = scipy.integrate.solve_ivp(rhs,[0,xmax],w0,t_eval=t_eval,atol=1.e-12,rtol=1.e-12)
    forwardsoln1 = scipy.integrate.solve_ivp(rhs,[xmax,2*xmax],w01,t_eval=t_eval1,atol=1.e-12,rtol=1.e-12)
    v = forwardsoln.y[0,:]
    x = forwardsoln.t
    v1 = forwardsoln1.y[0,:]
    x1 = forwardsoln1.t
    axes[0].scatter([-np.sqrt(mu/kappa),0,np.sqrt(mu/kappa)],[0,0,0],c='k',s=50)
    axes[1].plot(x,v,'-r',lw=3)
    axes[1].plot(x1,v1,'-r',lw=3)
    axes[0].plot(v[1:],np.diff(v)/np.diff(x),'--r',lw=3)
    axes[0].set_xlim(v.min(),v.max())
    # axes[0].plot(v1[1:],np.diff(v1)/np.diff(x1),'-r',lw=3)
    axes[1].set_xlim(0,xmax)
    axes[1].set_ylim(np.min(v),np.max(v))
    # axes[0].set_title('Phase plane')
    axes[0].set_xlabel(r'$\overline{q}_0$'); axes[0].set_ylabel(r'$\overline{q}_1$')
    axes[1].set_title('v(x)')
    axes[1].set_xlabel('x')
    #Unscaled axis 0
    axes[0].set_xlim(-1.5,1.5)
    axes[0].set_ylim(-1.,1.)
    axes[0].axis('image')
    fig.canvas.draw_idle()
    plt.close()
    return fig
    
interact(NLSH,v0=FloatSlider(min=-1.5,max=1.5,step=0.001,value=1.e-8),
         mu=FloatSlider(min=-10.,max=1.,step=0.01,value=1.),
         tau=FloatSlider(min=0.001,max=10.,step=0.001,value=0.1),
         xmax=FloatSlider(min=1,max=1000,step=1,value=20),
         kappa=FloatSlider(min=-10.,max=10,step=0.1,value=1),
         continuous_update=False);

interactive(children=(FloatSlider(value=1.0, description='mu', max=1.0, min=-10.0, step=0.01), FloatSlider(val…